In [ ]:
# Databricks notebook source# MAGIC %md# MAGIC # Medallion Architecture: Bronze Layer# MAGIC # MAGIC **Purpose:** Ingest raw data from mirrored SalesLT database into Bronze layer# MAGIC # MAGIC **Source:** `RvDSQL.MountedRelationalDatabase` (Mirrored Database)# MAGIC # MAGIC **Target:** `SalesAnalytics.Lakehouse` - Bronze layer tables# MAGIC # MAGIC **Pattern:** # MAGIC - Raw data copy with minimal transformation# MAGIC - Add audit columns: _ingest_timestamp, _source_file# MAGIC - Preserve all source columns and data types# MAGIC - Full load pattern (can be enhanced to incremental later)# COMMAND ----------# MAGIC %md# MAGIC ## Configuration# COMMAND ----------from pyspark.sql import SparkSessionfrom pyspark.sql.functions import *from pyspark.sql.types import *from datetime import datetimeimport logging# Configure logginglogging.basicConfig(level=logging.INFO)logger = logging.getLogger(__name__)# ConfigurationWORKSPACE_ID = "SalesLT"SOURCE_ITEM = "RvDSQL.MountedRelationalDatabase"TARGET_LAKEHOUSE = "SalesAnalytics"SOURCE_SCHEMA = "SalesLT"BRONZE_LAYER = "bronze"# Tables to ingestBRONZE_TABLES = [    "Customer",    "Product",    "ProductCategory",    "ProductModel",    "ProductDescription",    "ProductModelProductDescription",    "Address",    "CustomerAddress",    "SalesOrderHeader",    "SalesOrderDetail"]def get_source_path(table_name):    return f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SOURCE_ITEM}/Tables/{SOURCE_SCHEMA}/{table_name}"def get_bronze_table_name(source_table):    return f"{BRONZE_LAYER}_{source_table}"logger.info(f"Bronze layer configuration loaded. Will ingest {len(BRONZE_TABLES)} tables.")# COMMAND ----------# MAGIC %md# MAGIC ## Bronze Ingestion Function# COMMAND ----------def ingest_to_bronze(source_table_name):    """    Ingest a table from mirrored database to Bronze layer        Pattern:    1. Read from source (mirrored database)    2. Add audit columns    3. Write to Bronze as Delta table    """    logger.info(f"Starting Bronze ingestion for {source_table_name}...")    start_time = datetime.now()        try:        # Read source table        source_path = get_source_path(source_table_name)        df = spark.read.format("delta").load(source_path)                source_count = df.count()        logger.info(f"  Read {source_count:,} rows from source {source_table_name}")                # Add audit columns        bronze_df = df \            .withColumn("_ingest_timestamp", current_timestamp()) \            .withColumn("_source_system", lit("RvDSQL.MountedRelationalDatabase")) \            .withColumn("_source_schema", lit(SOURCE_SCHEMA)) \            .withColumn("_source_table", lit(source_table_name))                # Write to Bronze layer        bronze_table_name = get_bronze_table_name(source_table_name)        bronze_df.write \            .format("delta") \            .mode("overwrite") \            .option("overwriteSchema", "true") \            .saveAsTable(f"{TARGET_LAKEHOUSE}.{bronze_table_name}")                # Log completion        duration = (datetime.now() - start_time).total_seconds()        logger.info(f"✓ {bronze_table_name}: {source_count:,} rows ingested in {duration:.2f}s")                return {            "table": bronze_table_name,            "status": "success",            "rows": source_count,            "duration_seconds": duration        }            except Exception as e:        logger.error(f"✗ Failed to ingest {source_table_name}: {str(e)}")        return {            "table": source_table_name,            "status": "failed",            "error": str(e)        }# COMMAND ----------# MAGIC %md# MAGIC ## Execute Bronze Ingestion# COMMAND ----------# Ingest all tableslogger.info("=" * 70)logger.info("BRONZE LAYER INGESTION")logger.info("=" * 70)ingestion_results = []total_start = datetime.now()for table in BRONZE_TABLES:    result = ingest_to_bronze(table)    ingestion_results.append(result)total_duration = (datetime.now() - total_start).total_seconds()# Summarylogger.info("\n" + "=" * 70)logger.info("BRONZE INGESTION SUMMARY")logger.info("=" * 70)successful = [r for r in ingestion_results if r["status"] == "success"]failed = [r for r in ingestion_results if r["status"] == "failed"]logger.info(f"Total tables processed: {len(BRONZE_TABLES)}")logger.info(f"Successful: {len(successful)}")logger.info(f"Failed: {len(failed)}")logger.info(f"Total duration: {total_duration:.2f}s")if successful:    logger.info("\nSuccessful ingestions:")    total_rows = sum(r["rows"] for r in successful)    for result in successful:        logger.info(f"  ✓ {result['table']}: {result['rows']:,} rows")    logger.info(f"\nTotal rows ingested to Bronze: {total_rows:,}")if failed:    logger.info("\nFailed ingestions:")    for result in failed:        logger.info(f"  ✗ {result['table']}: {result['error']}")# COMMAND ----------# MAGIC %md# MAGIC ## Bronze Layer Validation# COMMAND ----------# Verify all Bronze tables exist and show row countslogger.info("\nBRONZE LAYER VERIFICATION")logger.info("-" * 70)for table in BRONZE_TABLES:    bronze_table = get_bronze_table_name(table)    try:        df = spark.read.table(f"{TARGET_LAKEHOUSE}.{bronze_table}")        count = df.count()        schema_count = len(df.columns)        logger.info(f"{bronze_table}: {count:,} rows, {schema_count} columns")    except Exception as e:        logger.error(f"{bronze_table}: NOT FOUND or ERROR - {str(e)}")logger.info("\n✓ Bronze layer ingestion complete!")